# Emerging Technologies Assessment

**Student:** Hasan Murtaza 


## Problem 1: Generating Random Boolean Functions

### Understanding the Requirements

The [Deutsch-Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa) works with functions that:
- Accept a fixed number of Boolean inputs (in this case: 4)
- Return a single Boolean output
- Are guaranteed to be either **constant** or **balanced**

With 4 Boolean inputs, we have 2^4 = **16 possible input combinations**.

### Constant vs Balanced Functions

**Constant functions:** Always return the same value
- Always return `False` (0), OR
- Always return `True` (1)
- Total: **2 constant functions**

**Balanced functions:** Return `True` for exactly half the inputs
- Return `True` for exactly 8 of the 16 inputs
- Return `False` for the other 8 inputs
- Total: C(16,8) = **12,870 balanced functions**

In [6]:
# Import required libraries
import random
import numpy as np

### Truth Table Representation

Any Boolean function can be represented as a truth table. For a 4-input function:

| Input (binary) | Decimal | Output |
|---------------|---------|--------|
| 0000 | 0 | f(0,0,0,0) |
| 0001 | 1 | f(0,0,0,1) |
| ... | ... | ... |
| 1111 | 15 | f(1,1,1,1) |

We can store the output column as a **tuple of length 16**, where each position corresponds to the decimal value of the binary input.

### Approach

Based on the [notes on random functions](https://github.com/ianmcloughlin/emerging-technologies/blob/main/notes/random-functions.ipynb), I will:

1. Generate a random output tuple (constant or balanced)
2. Create a closure that uses this tuple
3. The closure converts 4 Boolean inputs to an integer index
4. Return the value at that index in the tuple

This approach hides the function's internal structure, making it suitable for testing the Deutsch-Jozsa algorithm.

### Helper Function 1: Generating Random Tuples

The First helper function needed is a function that generates a random tuple representing either a constant or balanced function.

In [7]:
def random_tuple(n):
    """
    Generate a random constant or balanced tuple of length 2**n.
    
    Parameters:
        n (int): Number of input bits
    
    Returns:
        tuple: Length 2**n, either all same value (constant) or half 0s/1s (balanced)
    """
    # Randomly choose function type (50/50 probability)
    ftype = random.choice(['constant', 'balanced'])
    
    if ftype == 'constant':
        # Choose all 0s or all 1s
        zero_or_one = random.choice([0, 1])
        # Create tuple by repeating the value 2**n times
        t = (zero_or_one,) * (2**n)
    else:  # balanced
        # Create list with half 0s, half 1s
        t = ([0] * (2**(n-1))) + ([1] * (2**(n-1)))
        # Shuffle to randomize positions
        random.shuffle(t)
        # Convert to immutable tuple
        t = tuple(t)
    
    return t

### Understanding Tuple Multiplication

The expression `(value,) * n` creates a tuple by repeating `value` n times. The comma after `value` is crucial, it makes it a tuple, not just a parenthesized expression.

In [8]:
# Example: Create tuple of 8 zeros
zeros = (0,) * 8
print(f"Eight zeros: {zeros}")

# Example: Create tuple of 8 ones  
ones = (1,) * 8
print(f"Eight ones: {ones}") 

Eight zeros: (0, 0, 0, 0, 0, 0, 0, 0)
Eight ones: (1, 1, 1, 1, 1, 1, 1, 1)


### Testing random_tuple



In [9]:
# Generate some random tuples for n=4 (16 elements)
print("Random tuples for n=4:")
for i in range(5):
    t = random_tuple(4)
    # Count ones to check if constant or balanced
    ones_count = sum(t)
    if ones_count == 0 or ones_count == 16:
        ftype = "constant"
    else:
        ftype = "balanced"
    print(f"{t} - {ftype} ({ones_count} ones)")

Random tuples for n=4:
(1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1) - constant (16 ones)
(1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1) - constant (16 ones)
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0) - constant (0 ones)
(1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1) - constant (16 ones)
(0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1) - balanced (8 ones)


### Helper Function 2: Converting Binary Arguments to Integer

We need to convert 4 Boolean arguments (e.g., `1, 0, 1, 1`) into an integer index (e.g., 11 in decimal = 1011 in binary).

Python's **variadic function** syntax (`*args`) allows us to accept any number of arguments. See [Real Python's guide on *args](https://realpython.com/python-boolean/) for more details.

In [10]:
def bin_args_to_int(*args):
    """
    Convert binary arguments to an integer.
    
    Parameters:
        *args: Variable number of binary values (0/1 or False/True)
    
    Returns:
        int: Decimal representation of the binary input
    """
    # Convert any truthy values to '1', falsy to '0'
    bits = ['1' if i else '0' for i in args]
    # Join into binary string
    bits_str = ''.join(bits)
    # Convert binary string to integer (base 2)
    return int(bits_str, 2)

### Testing Binary Conversion

Verify this works correctly:

In [11]:
# Test binary to integer conversion
print(f"Binary 0000 = {bin_args_to_int(0, 0, 0, 0)}")
print(f"Binary 0001 = {bin_args_to_int(0, 0, 0, 1)}")
print(f"Binary 1011 = {bin_args_to_int(1, 0, 1, 1)}")
print(f"Binary 1111 = {bin_args_to_int(1, 1, 1, 1)}")

Binary 0000 = 0
Binary 0001 = 1
Binary 1011 = 11
Binary 1111 = 15
